# ⚖️ Interprétabilité & Analyse des Biais

**Projet 3 — Détection automatique de Fake News politiques**

Ce notebook répond aux 3 questions éthiques du projet :
1. **Le modèle peut-il favoriser ou pénaliser certains groupes ?**
2. **Les erreurs pourraient-elles avoir des conséquences importantes ?**
3. **Peut-on détecter la désinformation uniquement à partir du texte ?**

Outils utilisés :
- **LIME** : explications locales des prédictions
- **Top features TF-IDF** : features globaux discriminants
- **Analyse par parti et speaker** : détection des biais

## 0. Imports & Configuration

In [ ]:
import os, re, json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from lime.lime_text import LimeTextExplainer
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
STOP_WORDS = set(stopwords.words('english'))
SEED = 42
LABEL_ORDER  = ['pants-fire','false','barely-true','half-true','mostly-true','true']
LABEL_COLORS = ['#E24B4A','#F09595','#FAC775','#97C459','#5DCAA5','#1D9E75']

print(' Imports OK')

## 1. Chargement des données & modèles

In [ ]:
tfidf    = joblib.load('../data/modeles/tfidf_vectorizer.joblib')
lr_model = joblib.load('../data/modeles/baseline_logreg.joblib')

test_df  = pd.read_parquet('../data/traitees/liar_base_clean.parquet')
train_df = pd.read_parquet('../data/traitees/liar_train.parquet')

# Recalcul pred_lr si absent
if 'pred_lr' not in test_df.columns:
    test_df['pred_lr'] = lr_model.predict(
        tfidf.transform(test_df['enriched_text'].fillna(''))
    )

print(f'Test : {len(test_df):,} déclarations')
print(f' Modèles chargés')

## 2. Interprétabilité — LIME

LIME (Local Interpretable Model-agnostic Explanations) explique **pourquoi** le modèle prédit `fake` ou `real` pour une déclaration spécifique.

In [ ]:
def predict_proba(texts):
    return lr_model.predict_proba(tfidf.transform(texts))

explainer = LimeTextExplainer(class_names=['fake', 'real'])

# Sélection des 4 cas
cases = {
    'correct_fake': test_df[(test_df['binary_label']==0) & (test_df['pred_lr']==0)].index,
    'correct_real': test_df[(test_df['binary_label']==1) & (test_df['pred_lr']==1)].index,
    'error_fake':   test_df[(test_df['binary_label']==0) & (test_df['pred_lr']==1)].index,
    'error_real':   test_df[(test_df['binary_label']==1) & (test_df['pred_lr']==0)].index,
}

print(f'Cas disponibles :')
for k, v in cases.items():
    print(f'  {k} : {len(v)} exemples')

In [ ]:
for case_name, idx_list in cases.items():
    if len(idx_list) == 0:
        continue
    idx      = idx_list[0]
    row      = test_df.loc[idx]
    text     = row['enriched_text']
    true_lbl = 'fake' if row['binary_label']==0 else 'real'
    pred_lbl = 'fake' if row['pred_lr']==0 else 'real'
    icon     = '✅' if true_lbl == pred_lbl else '❌'

    print(f'\n{icon} {case_name.upper()}')
    print(f'   Déclaration : {str(row.get("statement",""))[:90]}...')
    print(f'   Speaker : {row.get("speaker","N/A")} | Parti : {row.get("party","N/A")}')
    print(f'   Vrai : {true_lbl} | Prédit : {pred_lbl}')

    exp = explainer.explain_instance(
        text, predict_proba, num_features=10, num_samples=500
    )
    print('   Mots influents :')
    for word, weight in exp.as_list():
        sym = ' →fake' if weight < 0 else ' →real'
        print(f'     {sym}  "{word}"  ({weight:+.4f})')

    out_html = f'../Doc/LIME_{case_name}.html'
    exp.save_to_file(out_html)
    print(f'    HTML : {out_html}')

In [ ]:
# Top features globaux
feature_names = np.array(tfidf.get_feature_names_out())
coef  = lr_model.coef_[0]
top_n = 25

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
top_fake_idx = np.argsort(coef)[:top_n][::-1]
top_real_idx = np.argsort(coef)[-top_n:][::-1]

axes[0].barh(feature_names[top_fake_idx][::-1],
             np.abs(coef[top_fake_idx][::-1]), color='#E24B4A', edgecolor='white')
axes[0].set_title(f'Top {top_n} mots → FAKE', fontweight='bold', color='#A32D2D')
axes[0].set_xlabel('|Coefficient|')

axes[1].barh(feature_names[top_real_idx][::-1],
             np.abs(coef[top_real_idx][::-1]), color='#1D9E75', edgecolor='white')
axes[1].set_title(f'Top {top_n} mots → REAL', fontweight='bold', color='#0F6E56')
axes[1].set_xlabel('|Coefficient|')

plt.suptitle('Features discriminants — Logistic Regression (TF-IDF)', fontweight='bold')
plt.tight_layout()
plt.savefig('../Doc/BIAS_01_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Analyse des biais — Par parti politique

In [ ]:
top_parties  = test_df['party'].value_counts().head(8).index
party_results = []

for party in top_parties:
    sub = test_df[test_df['party'] == party]
    if len(sub) < 5: continue
    f1_lr    = f1_score(sub['binary_label'], sub['pred_lr'], average='macro', zero_division=0)
    fake_rate= (sub['binary_label']==0).mean() * 100
    party_results.append({
        'Parti': party, 'N': len(sub),
        'Taux fake (%)': round(fake_rate, 1),
        'F1 LR': round(f1_lr, 3)
    })

bias_df = pd.DataFrame(party_results).sort_values('F1 LR', ascending=False)
print(' Performances par parti politique :\n')
display(bias_df)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(bias_df['Parti'], bias_df['F1 LR'], color='#7F77DD', edgecolor='white')
axes[0].set_ylim(0, 1)
axes[0].set_title('F1 macro par parti politique', fontweight='bold')
axes[0].set_ylabel('F1 macro')
axes[0].tick_params(axis='x', rotation=35)

cols = ['#E24B4A' if r>60 else '#EF9F27' if r>40 else '#1D9E75'
        for r in bias_df['Taux fake (%)']]
axes[1].bar(bias_df['Parti'], bias_df['Taux fake (%)'], color=cols, edgecolor='white')
axes[1].axhline(50, color='gray', linestyle='--', alpha=0.7)
axes[1].set_title('Taux fake par parti (%)', fontweight='bold')
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis='x', rotation=35)

plt.suptitle('Analyse des biais partisans', fontweight='bold')
plt.tight_layout()
plt.savefig('../Doc/BIAS_02_party_bias.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Taux fake par speaker
spk = (
    test_df.groupby('speaker')['binary_label']
    .agg(total='count', fake_count=lambda x: (x==0).sum())
    .query('total >= 5')
    .copy()
)
spk['fake_rate'] = spk['fake_count'] / spk['total']
spk = spk.sort_values('fake_rate', ascending=False).head(12)

plt.figure(figsize=(13, 5))
cols = ['#E24B4A' if r>0.65 else '#EF9F27' if r>0.45 else '#1D9E75'
        for r in spk['fake_rate']]
bars = plt.bar(spk.index, spk['fake_rate'], color=cols, edgecolor='white')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.7, label='Seuil 50%')
for bar, val, n in zip(bars, spk['fake_rate'], spk['total']):
    plt.text(bar.get_x()+bar.get_width()/2, val+0.01,
             f'{val:.0%}\n(n={n})', ha='center', fontsize=8)
plt.title('Taux de fausses déclarations par speaker (n≥5)', fontweight='bold')
plt.ylabel('Taux fake')
plt.xticks(rotation=40, ha='right')
plt.ylim(0, 1.1)
plt.legend()
plt.tight_layout()
plt.savefig('../Doc/BIAS_03_speaker_bias.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Test : impact des métadonnées (biais speaker)

In [ ]:
import re

def clean_text(text):
    if pd.isna(text): return ''
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', '', text)
    return ' '.join(w for w in text.split() if w not in STOP_WORDS and len(w)>1)

# Modèle sans métadonnées
tfidf_raw = TfidfVectorizer(max_features=15000, ngram_range=(1,2),
                             sublinear_tf=True, min_df=2)
X_tr_raw  = tfidf_raw.fit_transform(train_df['statement'].apply(clean_text))
X_te_raw  = tfidf_raw.transform(test_df['statement'].apply(clean_text))

lr_raw = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr_raw.fit(X_tr_raw, train_df['binary_label'])
preds_raw = lr_raw.predict(X_te_raw)

f1_raw = f1_score(test_df['binary_label'], preds_raw, average='macro')
f1_enr = f1_score(test_df['binary_label'], test_df['pred_lr'], average='macro')

print(' Impact des métadonnées (speaker + parti) :')
print(f'  Sans métadonnées : F1 = {f1_raw:.3f}')
print(f'  Avec métadonnées : F1 = {f1_enr:.3f}  (Δ = {f1_enr-f1_raw:+.3f})')

if abs(f1_enr - f1_raw) > 0.03:
    print('\n    Biais speaker significatif détecté :')
    print('     Le modèle utilise partiellement l\'identité du locuteur,')
    print('     et pas uniquement le contenu factuel de la déclaration.')
else:
    print('\n   Impact modéré — le texte reste le signal principal.')

## 5. Discussion éthique

### ① Biais liés aux sources des données
Les annotations proviennent de **PolitiFact**, un fact-checker américain. Ses choix éditoriaux sur les sujets ou personnalités couverts peuvent introduire un biais systématique.

### ② Biais liés aux speakers
Comme montré ci-dessus, enrichir le texte avec l'identité du speaker améliore le F1 mais crée un **biais de réputation** : le modèle peut classer une déclaration comme `fake` uniquement parce qu'elle est attribuée à un speaker connu pour ses mensonges, indépendamment du contenu.

### ③ Biais d'annotation humaine
Les 6 labels de véracité sont subjectifs. Deux annotateurs peuvent classer la même déclaration différemment, en particulier pour `barely-true` et `half-true`. Le mapping en 2 classes accentue ces approximations.

### ④ Limites intrinsèques du texte seul
Certaines déclarations **ne peuvent pas** être vérifiées par analyse textuelle seule :
- `"Le taux de chômage était de 4.2% en 2015"` → nécessite des données économiques
- `"Notre allié a trahi sa promesse"` → nécessite un contexte diplomatique

Le modèle détecte des **patterns linguistiques associés à la fausseté**, pas des faits objectifs.

### ⑤ Conséquences réelles des erreurs
| Erreur | Conséquence |
|--------|-------------|
| Faux positif (real → fake) | Censure injustifiée de contenu légitime |
| Faux négatif (fake → real) | Propagation non détectée de désinformation |

In [ ]:
# Analyse quantitative des erreurs
errors  = test_df[test_df['binary_label'] != test_df['pred_lr']].copy()
correct = test_df[test_df['binary_label'] == test_df['pred_lr']].copy()

print(f' Analyse des erreurs (Logistic Regression) :')
print(f'  Corrects : {len(correct)} ({len(correct)/len(test_df)*100:.1f}%)')
print(f'  Erreurs  : {len(errors)} ({len(errors)/len(test_df)*100:.1f}%)')
print(f'\n  Erreurs par label original :')
err_by_label = errors['label'].value_counts().reindex(LABEL_ORDER).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(err_by_label.index, err_by_label.values, color=LABEL_COLORS, edgecolor='white')
axes[0].set_title('Erreurs par label original', fontweight='bold')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Nombre d\'erreurs')
axes[0].tick_params(axis='x', rotation=25)

errors['text_len']  = errors['enriched_text'].apply(lambda x: len(str(x).split()))
correct['text_len'] = correct['enriched_text'].apply(lambda x: len(str(x).split()))
axes[1].hist(correct['text_len'], bins=30, alpha=0.7, color='#7F77DD',
             label=f'Correct (n={len(correct)})', edgecolor='white')
axes[1].hist(errors['text_len'],  bins=30, alpha=0.7, color='#E24B4A',
             label=f'Erreur (n={len(errors)})',   edgecolor='white')
axes[1].set_title('Longueur : corrects vs erreurs', fontweight='bold')
axes[1].set_xlabel('Nombre de mots')
axes[1].legend()

plt.tight_layout()
plt.savefig('../Doc/BIAS_04_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
bias_df.to_json('../data/modeles/bias_by_party.json', orient='records', indent=2)

print(' Doc/LIME_*.html — explications interactives (ouvrir dans navigateur)')
print(' Doc/BIAS_*.png — visualisations biais')
print(' data/modeles/bias_by_party.json')
print('\n Interprétabilité & Biais terminés — Projet complet !')